# GameTheory-03d — Deux espèces de flèches : quand une transformation est-elle aussi un morphisme ?
**Navigation** : [<< 3b-Chambres-Murs-Chemins](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb) | [Index](README.md) | [Version Python](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) | [3e-Meta-Actions-Tarifees >>](GameTheory-03e-Meta-Actions-Tarifees-Python.ipynb)


**Objectifs de ce notebook :**

1. **Distinguer deux espèces de flèches** qui coexistent sur l'espace des jeux 2x2 ordinaux : les flèches qui *transforment* la structure (les swaps de rangs de Robinson-Goforth) et les flèches qui la *préservent* (les morphismes de jeux au sens Tohmé-Viglizzo).
2. **Rendre la question finie et décidable** : sur les 576 jeux stricts et les 6 générateurs adjacents, la question « ce générateur est-il un morphisme pour ce jeu ? » est une énumération de 3456 paires — pas un slogan.
3. **Exhiber un contre-exemple concret** où un swap détruit un équilibre de Nash qu'un morphisme aurait préservé.
4. **Énumérer l'intersection** : combien de paires (jeu, générateur) le générateur préserve-t-il ? Le compte doit être reproductible.
5. **Caractériser** la préservation par une condition simple sur le jeu de départ — et la vérifier exhaustivement, pas seulement la conjecturer.

Ce notebook est la première expérience de la « Loi III — les deux espèces de flèches » sur un univers fini où tout se vérifie. Il prolonge [`GameTheory-03-Topology2x2-Python.ipynb`](GameTheory-03-Topology2x2-Python.ipynb), dont il réutilise la représentation ordinale.

**Prérequis** : jeux 2×2 et équilibres purs. **Durée** : 2 h.

## 1. Le problème : deux espèces de flèches sur le même espace

La théorie des jeux dispose de deux traditions qui font circuler des flèches entre les jeux, et **elles ne font pas la même chose** :

| Espèce | Ce qu'elle fait | Source |
|---|---|---|
| **Flèches qui PRÉSERVENT la structure** | morphismes de jeux : ils transportent les équilibres d'un jeu vers un autre jeu qui « joue pareil » | catégories concrètes de jeux de Tohmé-Viglizzo (`Gam_I`, `Gam`) |
| **Flèches qui TRANSFORMENT la structure** | swaps de rangs : ils changent les préférences d'un joueur, donc le jeu lui-même | topologie des jeux 2x2 de Robinson-Goforth |

Un **swap de rangs** échange deux valeurs dans la table de gains d'un joueur — par exemple, permuter les rangs 1 et 2 du joueur Ligne. Ce n'est pas un changement de croyances ni d'actions : c'est un **changement des préférences elles-mêmes**, donc du monde descriptif. Un **morphisme de jeux**, à l'inverse, garantit que ce qui est rationnel ici l'est encore là-bas : les équilibres se transportent.

La **Loi III** du chantier 4 s'énonce alors précisément :

> Un swap change les préférences — il n'est donc **pas automatiquement** un morphisme. La question n'est pas « transformation ou morphisme ? » en général, mais **« pour ce jeu précis, ce générateur précis, préserve-t-il les équilibres ? »**.

Et cette question, sur un univers fini, a une réponse mécanique.

### La question rendue finie

L'univers de `GameTheory-3` est l'espace des **jeux 2x2 stricts ordinaux** : chaque joueur distribue les rangs 1-4 sur les quatre cases. Il y a $24 \times 24 = 576$ tels jeux. La grammaire de Robinson-Goforth les fait bouger par **six générateurs adjacents** :

- $R_{12}, R_{23}, R_{34}$ : échanger les rangs adjacents $(1,2)$, $(2,3)$ ou $(3,4)$ dans les gains du joueur Ligne ;
- $C_{12}, C_{23}, C_{34}$ : la même chose pour le joueur Colonne.

Ces six swaps suffisent à engendrer tout le groupe $S_4 \times S_4$ des permutations de rangs (les transpositions adjacentes engendrent $S_4$) : un jeu ne se décrit pas, **il se rejoint** depuis un autre.

Sur cet espace, la question « transformation ou morphisme ? » devient une propriété **décidable par énumération** :

$$576 \text{ jeux} \times 6 \text{ générateurs} = 3456 \text{ paires à trancher}.$$

Réduction honnête de la notion de morphisme : dans les catégories de Tohmé-Viglizzo, un morphisme de jeux transporte aussi les stratégies. Ici, un swap **ne déplace aucune stratégie** — seule l'étiquette des gains change — donc l'application induite sur les profils est l'identité. L'espèce préservante se teste alors opérationnellement :

> **Définition (finie).** Le générateur $g$ est un *morphisme pour le jeu $G$* si $g$ préserve l'ensemble des équilibres de Nash purs : $\text{Nash}(G) = \text{Nash}(g(G))$, la comparaison se faisant cellule à cellule (application identité).

C'est cette définition que tout le notebook met à l'épreuve.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
record Game {
 public int[] R { get; } public int[] C { get; } public string Name { get; }
 public Game(int[] r,int[] c,string name="") {
  if(r==null || c==null || !r.Order().SequenceEqual(new[]{1,2,3,4}) || !c.Order().SequenceEqual(new[]{1,2,3,4}))
   throw new ArgumentException("Chaque table doit être une permutation des rangs 1 à 4.");
  R=(int[])r.Clone();C=(int[])c.Clone();Name=name;
 }
}
var classic=new Dictionary<string,Game> {
 ["Prisoner's Dilemma"]=new(new[]{3,1,4,2},new[]{3,4,1,2},"Prisoner's Dilemma"),
 ["Stag Hunt"]=new(new[]{4,1,3,2},new[]{4,3,1,2},"Stag Hunt"),
 ["Chicken"]=new(new[]{3,2,4,1},new[]{3,4,2,1},"Chicken"),
 ["Matching Pennies"]=new(new[]{4,1,2,3},new[]{1,4,3,2},"Matching Pennies"),
 ["Deadlock"]=new(new[]{2,1,4,3},new[]{2,4,1,3},"Deadlock"),
 ["Harmony"]=new(new[]{4,3,2,1},new[]{4,2,3,1},"Harmony")};
void Show(Game g) {Console.WriteLine($"{g.Name}\n  ({g.R[0]},{g.C[0]}) ({g.R[1]},{g.C[1]})\n  ({g.R[2]},{g.C[2]}) ({g.R[3]},{g.C[3]})");}
Console.WriteLine($".NET {Environment.Version} ; six jeux classiques :");foreach(var g in classic.Values)Show(g);

.NET 10.0.14 ; six jeux classiques :


Prisoner's Dilemma
  (3,3) (1,4)
  (4,1) (2,2)


Stag Hunt
  (4,4) (1,3)
  (3,1) (2,2)


Chicken
  (3,3) (2,4)
  (4,2) (1,1)


Matching Pennies
  (4,1) (1,4)
  (2,3) (3,2)


Deadlock
  (2,2) (1,4)
  (4,1) (3,3)


Harmony
  (4,4) (3,2)
  (2,3) (1,1)


### Lecture de la représentation

Chaque case affiche `(gain Ligne, gain Colonne)` en rangs ordinaux — **le rang 4 est le meilleur** pour le joueur qui le reçoit. Deux observations qui serviront tout le long :

- **Lecture par colonne** : dans la colonne $C_j$, le joueur Ligne compare ses deux gains ; le plus grand désigne sa **meilleure réponse** dans cette colonne. Symétriquement, le joueur Colonne se lit **par ligne**.
- **Stricteté** : chaque joueur place exactement une fois chaque rang 1-4 ; il n'y a **aucune égalité**, donc chaque colonne (resp. ligne) a une meilleure réponse unique de chaque côté. C'est ce qui rend l'espace fini et propre — les jeux à égalité (les « murs » du grain D2, issue #12213) sont un autre objet.

Le Dilemme du Prisonnier se relit ainsi : pour Ligne, la colonne $C_0$ compare 3 et 4 (il préfère $R_1$ : déserter), la colonne $C_1$ compare 1 et 2 (il préfère encore $R_1$) — $R_1$ est une stratégie dominante, et l'équilibre unique $(R_1, C_1)$ est le pire outcome collectif.

In [2]:
int Nash(Game g) {int mask=0;for(int i=0;i<2;i++)for(int j=0;j<2;j++) {int k=2*i+j;if(g.R[k]>g.R[2*(1-i)+j] && g.C[k]>g.C[2*i+1-j])mask|=1<<k;}return mask;}
string Profiles(int mask)=>mask==0?"aucun":string.Join(", ",Enumerable.Range(0,4).Where(i=>(mask&(1<<i))!=0).Select(i=>$"(R{i/2},C{i%2})"));
foreach(var g in classic.Values)Console.WriteLine($"{g.Name} : {System.Numerics.BitOperations.PopCount((uint)Nash(g))} équilibre(s) : {Profiles(Nash(g))}");

Prisoner's Dilemma : 1 équilibre(s) : (R1,C1)


Stag Hunt : 2 équilibre(s) : (R0,C0), (R1,C1)


Chicken : 2 équilibre(s) : (R0,C1), (R1,C0)


Matching Pennies : 0 équilibre(s) : aucun


Deadlock : 1 équilibre(s) : (R1,C1)


Harmony : 1 équilibre(s) : (R0,C0)


### Interprétation : la structure de Nash, signature du jeu

L'ensemble des équilibres de Nash purs est la **signature stratégique** d'un jeu — c'est précisément ce qu'un morphisme de jeux s'engage à transporter. La collection classique couvre les quatre situations : un unique équilibre à dominance (Dilemme, Deadlock, Harmony), deux équilibres de coordination ou d'anti-coordination (Stag Hunt, Chicken), et **aucun équilibre pur** (Matching Pennies, où l'instabilité est le jeu).

Ce qui va nous intéresser n'est pas la classification elle-même (c'est le sujet de GameTheory-3, §5-6) mais une question plus fine : **cette signature survit-elle au passage d'un générateur ?** Un jeu sans équilibre peut en acquérir un, un jeu à équilibre unique peut le voir déménager — ou tout perdre. Chacun de ces événements est une **destruction** au sens morphique : la flèche existe (le swap est toujours applicable), mais elle ne préserve pas.

In [3]:
int[] Swap(int[] values,int a,int b)=>values.Select(v=>v==a?b:v==b?a:v).ToArray();
Game Apply(Game g,string gen)=>gen[0]=='R'?new Game(Swap(g.R,gen[1]-'0',gen[2]-'0'),g.C):new Game(g.R,Swap(g.C,gen[1]-'0',gen[2]-'0'));
string[] generators={"R12","R23","R34","C12","C23","C34"};
var pd=classic["Prisoner's Dilemma"];Console.WriteLine($"Dilemme : Nash {Profiles(Nash(pd))} ; après R12 : {Profiles(Nash(Apply(pd,"R12")))}");Show(Apply(pd,"R12"));

Dilemme : Nash (R1,C1) ; après R12 : (R0,C1)



  (3,3) (2,4)
  (4,1) (1,2)


### Interprétation : la grammaire en action — et le soupçon

Le générateur $R_{12}$ a échangé les rangs 1 et 2 du joueur Ligne : la case $(R_0, C_1)$ est passée de 1 à 2 et la case $(R_1, C_1)$ de 2 à 1. Les **stratégies n'ont pas bougé**, les préférences ont changé — et la sortie mesure déjà le soupçon fondateur de la Loi III : dans la colonne $C_1$, Ligne comparait 1 et 2 (il préférait $R_1$) ; il compare maintenant 2 et 1 (il préfère $R_0$). **Sa meilleure réponse dans cette colonne s'est retournée**, et avec elle l'équilibre.

Ce seul exemple montre les deux espèces à l'œuvre : la flèche $R_{12}$ **existe** toujours (c'est une transformation valide de l'espace) mais, pour ce jeu précis, elle **ne préserve pas** la signature de Nash. Reste à savoir si c'est un accident du Dilemme ou une structure de l'espace entier — c'est l'objet des sections suivantes. La réponse tiendra en une condition d'une ligne sur la table de gains du jeu de départ.

## 2. L'espèce préservante : tester le morphisme, flèche par flèche

La définition finie de la section 1 se implémente en une fonction : $g$ est un morphisme pour $G$ si et seulement si les ensembles de Nash coïncident **avant et après** l'application de $g$. Notons ce que cette comparaison ne demande pas : elle ne demande pas que le jeu soit « pareil » (les gains diffèrent en général), ni que les équilibres soient aux mêmes cases pour toujours — elle demande l'égalité des deux ensembles pour **cette** paire $(G, g)$.

C'est bien une relation *locale* : le même générateur peut être morphisme pour un jeu et pas pour son voisin. L'espace des flèches se partitionne donc en

- **flèches préservantes** — celles qui, pour ce jeu, transportent la signature de Nash intacte (l'identité sur les profils est alors compatible avec la structure de jeu) ;
- **flèches transformatrices** — celles qui cassent, déplacent ou créent des équilibres.

Et parce que l'univers est fini, chacune des 3456 paires tombe d'un côté ou de l'autre **sans zone grise**.

In [4]:
bool Preserves(Game g,string gen)=>Nash(g)==Nash(Apply(g,gen));
foreach(var (name,gen) in new[]{("Deadlock","R12"),("Deadlock","R34"),("Prisoner's Dilemma","R12"),("Prisoner's Dilemma","C12"),("Harmony","C34"),("Matching Pennies","R23")}) {var g=classic[name];Console.WriteLine($"{name} + {gen} : {Profiles(Nash(g))} -> {Profiles(Nash(Apply(g,gen)))} : {(Preserves(g,gen)?"PRESERVE":"TRANSFORME")}");}

Deadlock + R12 : (R1,C1) -> (R1,C1) : PRESERVE


Deadlock + R34 : (R1,C1) -> (R1,C1) : PRESERVE


Prisoner's Dilemma + R12 : (R1,C1) -> (R0,C1) : TRANSFORME


Prisoner's Dilemma + C12 : (R1,C1) -> (R1,C0) : TRANSFORME


Harmony + C34 : (R0,C0) -> (R0,C0) : PRESERVE


Matching Pennies + R23 : aucun -> aucun : PRESERVE


### Interprétation : deux espèces, même espace

Les six verdicts mesurés racontent déjà la Loi III :

- **Deadlock + $R_{12}$ : PRÉSERVÉ.** Pour Ligne, les colonnes comparent (2,4) et (1,3) — échanger 1 et 2 ne change aucun ordre interne de colonne : les meilleures réponses ne bougent pas, l'équilibre $(R_1, C_1)$ reste seul. La flèche transforme la *table* mais transporte la *structure* : elle est des deux espèces à la fois.
- **Prisoner's Dilemma + $R_{12}$ : TRANSFORMÉ.** La colonne $C_1$ comparait (1,2) ; après le swap elle compare (2,1) — la meilleure réponse de Ligne s'y retourne, et l'équilibre unique déménage de $(R_1, C_1)$ vers $(R_0, C_1)$.
- **Matching Pennies + $R_{23}$ : PRÉSERVÉ… trivialement.** L'ensemble vide est égal à lui-même : le jeu n'a pas d'équilibre pur avant ni après. La préservation ici ne dit rien d'intéressant — un cas limite qu'il faudra garder en tête en lisant les comptes de la section 4.

Le point structurant : **le verdict n'appartient ni au générateur ni au jeu seul, mais à la paire**. C'est exactement pourquoi la question « un swap est-il un morphisme ? » est mal posée sans son jeu — et pourquoi l'énumération qui suit n'est pas un exercice de style.

## 3. Exercice 1 : implémenter les deux espèces et produire un contre-exemple

À vous de jouer : la fonction `Counterexample` doit trouver, dans les jeux classiques, **une paire (jeu, générateur) où le swap détruit un équilibre** — c'est-à-dire où un équilibre de Nash du jeu de départ n'est plus équilibre dans le jeu d'arrivée. Un contre-exemple, pas un principe.

In [5]:
record Witness(string game,string gen,int before,int after,int destroyed);
List<Witness> Counterexample(Dictionary<string,Game> games) {
 // TODO : parcourir les six jeux et six générateurs.
 // Étape 1 : calculer Nash avant/après ; étape 2 : garder les équilibres perdus sans compensation.
 // Étape 3 : retourner jeu, générateur, Nash avant, après et profils détruits (masques).
 return null;
}

Console.WriteLine("Exercice à compléter");

Exercice à compléter


### Vérification de l’exercice
La garde contrôle la réponse sur les exemples et les jeux énumérés.


In [6]:
bool ValidCounterexample(Dictionary<string,Game> items,List<Witness> answer) => answer!=null && answer.Count>0 && answer.Distinct().Count()==answer.Count && answer.All(x=>items.ContainsKey(x.game) && generators.Contains(x.gen) && x.before==Nash(items[x.game]) && x.after==Nash(Apply(items[x.game],x.gen)) && x.destroyed==(x.before&~x.after) && x.destroyed!=0 && (x.after&~x.before)==0);
var ex1=Counterexample(classic);Console.WriteLine(ex1==null?"Exercice 1 à compléter":"Exercice 1, vérification : "+(ValidCounterexample(classic,ex1) && new[]{"Stag Hunt","Chicken"}.All(n=>{var item=new Dictionary<string,Game>{{n,classic[n]}};return ValidCounterexample(item,Counterexample(item));})));

Exercice 1 à compléter


### Exemple résolu : un équilibre détruit, un équilibre déplacé

La démonstration guidée ci-dessous tranche la question sur le Dilemme du Prisonnier avec les deux générateurs qui l'attaquent le plus naturellement : $R_{12}$ (Ligne) et $C_{12}$ (Colonne). On regarde les profils un par un — c'est la distinction destruction/déplacement qui compte :

- **déplacement** : l'équilibre de départ n'est plus là, mais un autre est apparu (la cardinalité est conservée) ;
- **destruction stricte** : un profil de Nash($G$) disparaît et aucun nouveau profil ne le compense.

In [7]:
foreach(var gen in new[]{"R12","C12","R34"}) {int before=Nash(pd),after=Nash(Apply(pd,gen));Console.WriteLine($"Dilemme + {gen} : {Profiles(before)} -> {Profiles(after)} ; perd {Profiles(before&~after)} ; crée {Profiles(after&~before)}");}
foreach(var g in classic.Values)foreach(var gen in generators) {int before=Nash(g),after=Nash(Apply(g,gen));if((before&~after)!=0 && (after&~before)==0)Console.WriteLine($"Destruction stricte : {g.Name} + {gen} perd {Profiles(before&~after)} sans compensation");}

Dilemme + R12 : (R1,C1) -> (R0,C1) ; perd (R1,C1) ; crée (R0,C1)


Dilemme + C12 : (R1,C1) -> (R1,C0) ; perd (R1,C1) ; crée (R1,C0)


Dilemme + R34 : (R1,C1) -> (R1,C1) ; perd aucun ; crée aucun


Destruction stricte : Stag Hunt + R12 perd (R1,C1) sans compensation


Destruction stricte : Stag Hunt + R34 perd (R0,C0) sans compensation


Destruction stricte : Stag Hunt + C12 perd (R1,C1) sans compensation


Destruction stricte : Stag Hunt + C34 perd (R0,C0) sans compensation


Destruction stricte : Chicken + R12 perd (R0,C1) sans compensation


Destruction stricte : Chicken + R34 perd (R1,C0) sans compensation


Destruction stricte : Chicken + C12 perd (R1,C0) sans compensation


Destruction stricte : Chicken + C34 perd (R0,C1) sans compensation


### Interprétation : trois régimes mesurés sur un seul jeu — et le mur habité

Le Dilemme exhibe d'un coup les **trois régimes** possibles d'une flèche :

- **$R_{12}$ et $C_{12}$ : déplacement.** L'équilibre unique déménage ($(R_1,C_1) \to (R_0,C_1)$ pour le premier, $\to (R_1,C_0)$ pour le second). L'ensemble change — pas morphisme — mais le jeu d'arrivée a toujours un équilibre : la richesse stratégique est transportée, pas amputée.
- **$R_{34}$ : préservation.** La colonne $C_0$ du Dilemme contient bien le couple adjacent $(3,4)$ pour Ligne — la flèche **traverse un mur** — et pourtant la signature de Nash ne bouge pas. Premier indice que la condition naïve « couple adjacent ⟹ transformation » est **fausse** : il faut comprendre pourquoi ce passage est invisible.
- Sur les autres classiques, **Stag Hunt et Chicken subissent des destructions strictes** : chacun de leurs deux équilibres peut être supprimé sans compensation par le bon générateur. C'est le contre-exemple dur que l'exercice 1 demande — une coordination qui **perd** une de ses sorties.

Le vocabulaire géométrique du chantier 4 s'impose : dans la lecture Bruns-Kimmich (grain D2, issue #12213), les jeux stricts sont des **chambres** et les swaps adjacents traversent des **murs**. Mais le régime $R_{34}$ du Dilemme montre la subtilité : **un mur ne change la chambre Nash que s'il est habité** — si aucune meilleure réponse de Colonne ne pointe vers la colonne traversée, aucun équilibre ne vivait contre ce mur, et le passage est stratégiquement invisible. C'est cette intuition que la section 6 transforme en théorème vérifié.

## 4. Énumération exhaustive : l'intersection des deux espèces

L'échantillon des classiques ne suffit pas : la Loi III demande le compte **sur l'espace entier**. L'énumération ci-dessous génère les 576 jeux stricts (toutes paires de permutations de 1-4), applique chacun des 6 générateurs à chacun des jeux, et tranche les 3456 paires une par une. Trois mesures en sortent :

1. le compte **par générateur** : combien de jeux chaque générateur préserve ;
2. la **distribution par jeu** : combien de générateurs transforment chaque jeu (de 0 à 6) ;
3. les **points fixes de la grammaire** : les jeux que les six générateurs préservent — les jeux « au cœur de leur chambre », inatteignables par un seul pas de la grammaire.

In [8]:
IEnumerable<int[]> Permute(int[] a,int k=0) {if(k==a.Length){yield return (int[])a.Clone();yield break;}for(int j=k;j<a.Length;j++){(a[k],a[j])=(a[j],a[k]);foreach(var p in Permute(a,k+1))yield return p;(a[k],a[j])=(a[j],a[k]);}}
var perms=Permute(new[]{1,2,3,4}).ToArray();var games=(from r in perms from c in perms select new Game(r,c)).ToArray();
Console.WriteLine($"Jeux stricts générés : {games.Length} ({perms.Length} x {perms.Length})");
foreach(var gen in generators){int count=games.Count(g=>Preserves(g,gen));Console.WriteLine($"{gen} : {count} préservés / {games.Length-count} transformés");}
var dist=games.GroupBy(g=>generators.Count(gen=>!Preserves(g,gen))).OrderBy(g=>g.Key);
foreach(var group in dist)Console.WriteLine($"{group.Key} générateur(s) transformateur(s) : {group.Count()} jeux");
Console.WriteLine($"TOTAL : {games.Sum(g=>generators.Count(gen=>Preserves(g,gen)))} sur {games.Length*generators.Length} ; points fixes : {games.Count(g=>generators.All(gen=>Preserves(g,gen)))}");

Jeux stricts générés : 576 (24 x 24)


R12 : 432 préservés / 144 transformés


R23 : 432 préservés / 144 transformés


R34 : 432 préservés / 144 transformés


C12 : 432 préservés / 144 transformés


C23 : 432 préservés / 144 transformés


... (3 outputs left out)

2 générateur(s) transformateur(s) : 180 jeux


3 générateur(s) transformateur(s) : 80 jeux


4 générateur(s) transformateur(s) : 16 jeux


TOTAL : 2592 sur 3456 ; points fixes : 100


### Interprétation : la séparation nette existe — et elle est massive

Les comptes mesurés établissent la Loi III sur l'univers entier :

- **Chaque générateur préserve 432 jeux sur 576 (75 %) et en transforme 144 (25 %).** Un quart de l'espace change de signature de Nash à chaque pas de la grammaire — la séparation n'est ni marginale ni décorative. Et aucun générateur n'est « globalement » morphisme : l'espèce d'une flèche est une propriété de la paire.
- **La distribution par jeu est unanime** : 100 jeux sont préservés par les six générateurs — les « points fixes de la grammaire », et 16 jeux sont transformés par **quatre** générateurs sur six — les plus exposés de l'espace. La masse (460 jeux) se concentre entre un et trois générateurs transformateurs.
- **2592 paires préservantes sur 3456** — dont les préservations triviales (les jeux sans équilibre pur qui n'en acquièrent pas après le swap) et les préservations *invisibles* découvertes en section 3 ($R_{34}$ sur le Dilemme) : le compte honnête les inclut, la lecture doit s'en souvenir.

Le verdict du grain D5 s'écrit : **la séparation nette apparaît** — certaines flèches préservent, d'autres transforment, et le partage n'est ni uniforme ni arbitraire. La distribution a une forme ; une forme annonce une cause. La section suivante commence par se tromper dessus — c'est voulu.

## 5. Exercice 2 : cartographier l'intersection par structure de Nash

Le compte global dit *combien* ; à vous de dire *quoi*. La fonction `MapIntersection` doit croiser l'énumération avec la **structure de Nash** des jeux de départ (nombre d'équilibres purs) pour répondre : les jeux préservés par un générateur sont-ils surtout des jeux sans équilibre (préservation triviale) ou surtout des jeux à équilibre unique ? La réponse mesure à quel point la préservation mesurée en section 4 est *substantielle*.

In [9]:
Dictionary<int,(int preserve,int transforme)> MapIntersection(IEnumerable<Game> items) {
 // TODO : pour chaque jeu, compter les équilibres puis les six verdicts.
 // Étape 1 : grouper par nombre d'équilibres ; étape 2 : agréger préservations et transformations.
 return null;
}

Console.WriteLine("Exercice à compléter");

Exercice à compléter


### Vérification de l’exercice
La garde contrôle la réponse sur les exemples et les jeux énumérés.


In [10]:
bool ValidMap(IEnumerable<Game> items,Dictionary<int,(int preserve,int transforme)> answer) {
 if(answer==null)return false;var data=items.ToArray();var keys=data.Select(g=>(int)System.Numerics.BitOperations.PopCount((uint)Nash(g))).Distinct().ToHashSet();
 if(!answer.Keys.ToHashSet().SetEquals(keys))return false;
 foreach(int k in keys){var subset=data.Where(g=>System.Numerics.BitOperations.PopCount((uint)Nash(g))==k).ToArray();int yes=subset.Sum(g=>generators.Count(gen=>Preserves(g,gen)));if(answer[k]!=(yes,subset.Length*6-yes))return false;}return true;
}
var ex2=MapIntersection(games);Console.WriteLine(ex2==null?"Exercice 2 à compléter":"Exercice 2, vérification : "+(ValidMap(games,ex2) && ValidMap(classic.Values,MapIntersection(classic.Values))));

Exercice 2 à compléter


## 6. Caractérisation : quand un générateur préserve-t-il ?

L'énumération a une forme ; cherchons la cause. Premier pas sûr — la mécanique des meilleures réponses :

1. **Un swap adjacent $R_{(a,b)}$ ne peut retourner une meilleure réponse que dans une colonne qui contient les deux rangs échangés.** Dans une colonne donnée, la meilleure réponse de Ligne ne dépend que de l'**ordre** de ses deux gains ; échanger les valeurs $a$ et $b$ ne change aucun ordre… sauf dans une colonne dont les deux gains sont précisément $a$ et $b$ (l'ordre s'y inverse).

On serait tenté de conclure : *« couple $\{a,b\}$ présent dans une colonne ⟹ transformation. »* C'est la **conjecture naïve** — et elle est **fausse** : le régime $R_{34}$ du Dilemme (section 3) la contredit déjà. La cellule suivante la teste sur tout l'espace, montre ses désaccords, puis construit la bonne condition en regardant **pourquoi** certains retournements sont invisibles :

2. **Un retournement n'est visible dans l'ensemble de Nash que si la colonne traversée est aussi une colonne-maison pour l'autre joueur.** Le profil $(i, C_j)$ n'est équilibre que si $C_j$ est *aussi* la meilleure réponse de Colonne en ligne $i$. Si les meilleures réponses de Colonne (dans les deux lignes) pointent toutes **ailleurs** que vers $C_j$, aucun équilibre ne vit dans la colonne $C_j$ — ni avant ni après le retournement de Ligne : le mur est traversé, mais personne n'habitait contre.

D'où la **caractérisation attendue pour les swaps adjacents** : $R_{(a,b)}$ transforme $G$ si et seulement si une colonne des gains de Ligne contient $\{a, b\}$ **et** cette colonne est la meilleure réponse de Colonne dans au moins une ligne (symétrique pour $C_{(a,b)}$ : la ligne contenant $\{a,b\}$ doit être la meilleure réponse de Ligne dans au moins une colonne). L'univers étant fini, tout cela se **vérifie**.

In [11]:
int? RowColumnWithPair(Game g,int a,int b){for(int j=0;j<2;j++)if(new[]{g.R[j],g.R[2+j]}.Order().SequenceEqual(new[]{a,b}.Order()))return j;return null;}
int? ColRowWithPair(Game g,int a,int b){for(int i=0;i<2;i++)if(new[]{g.C[2*i],g.C[2*i+1]}.Order().SequenceEqual(new[]{a,b}.Order()))return i;return null;}
bool Naive(Game g,string gen){int a=gen[1]-'0',b=gen[2]-'0';return gen[0]=='R'?RowColumnWithPair(g,a,b)==null:ColRowWithPair(g,a,b)==null;}
bool Condition(Game g,string gen){int a=gen[1]-'0',b=gen[2]-'0';if(gen[0]=='R'){int? j=RowColumnWithPair(g,a,b);return j==null || Enumerable.Range(0,2).All(i=>(g.C[2*i]>g.C[2*i+1]?0:1)!=j);}int? row=ColRowWithPair(g,a,b);return row==null || Enumerable.Range(0,2).All(j=>(g.R[j]>g.R[2+j]?0:1)!=row);}
var allPairs=from g in games from gen in generators select(g,gen);
var naiveDiff=allPairs.Where(x=>Preserves(x.g,x.gen)!=Naive(x.g,x.gen)).ToArray();int wrong=allPairs.Count(x=>Preserves(x.g,x.gen)!=Condition(x.g,x.gen));
var example=naiveDiff[0];Console.WriteLine($"Conjecture naïve : {naiveDiff.Length} désaccords sur {games.Length*6} ; exemple R=[{string.Join(",",example.g.R)}], C=[{string.Join(",",example.g.C)}] + {example.gen}, Nash {Profiles(Nash(example.g))} -> {Profiles(Nash(Apply(example.g,example.gen)))}");
Console.WriteLine($"Condition adjacente : {games.Length*6-wrong} accords, {wrong} désaccords");
int noPair=perms.Count(p=>new[]{p[0],p[2]}.Order().SequenceEqual(new[]{1,2})==false && new[]{p[1],p[3]}.Order().SequenceEqual(new[]{1,2})==false);
Console.WriteLine($"Dérivation R12 : {noPair} x 24 + 8 x 6 = {noPair*24+8*6} ; mesuré {games.Count(g=>Preserves(g,"R12"))}");

Conjecture naïve : 288 désaccords sur 3456 ; exemple R=[1,2,3,4], C=[1,2,3,4] + C12, Nash (R1,C1) -> (R1,C1)


Condition adjacente : 3456 accords, 0 désaccords


Dérivation R12 : 16 x 24 + 8 x 6 = 432 ; mesuré 432


### Interprétation : le théorème vérifié — verdict explicite du grain D5

La scène s'est jouée en deux actes, et c'est elle-même un enseignement :

- **Acte 1 — la conjecture naïve meurt mesurée.** « Couple adjacent en colonne ⟹ transformation » échoue sur 288 paires : tous les cas de **murs inhabités**. Sur un univers fini, une conjecture ne se discute pas, elle se compte.
- **Acte 2 — la condition pour un swap adjacent prédit les 3456 verdicts sans exception.** $R_{(a,b)}$ transforme $G$ ssi la colonne qui contient $\{a,b\}$ dans les gains de Ligne est **aussi** une colonne de meilleure réponse de Colonne. Et le compte s'en **dérive** : $16 \times 24$ jeux sans mur + $8 \times 6$ jeux au mur inhabité $= 384 + 48 = 432$ — exactement la mesure de la section 4. Le compte n'est plus un constat, c'est une conséquence.

**Verdict de l'exercice 3 de l'issue (explicité) :** oui, il existe une condition simple sur le jeu de départ qui prédit la préservation. Elle n'est pas *purement* une condition sur le joueur visé (comme le suggérait l'intuition) : elle **couple les deux tables** — la géométrie des rangs d'un côté, la direction des meilleures réponses de l'autre. C'est plus riche que prévu, et c'est vérifié exhaustivement.

Relue en langage géométrique : **traverser un mur ne change de chambre Nash que si le mur est habité** — si aucune meilleure réponse de l'autre joueur ne pointe vers la colonne traversée, aucun équilibre ne vivait contre ce mur et le passage est stratégiquement invisible. La grammaire (swaps de rangs) et la géométrie (chambres/murs du grain D2) racontent la même histoire : les transitions se jouent *là où les préférences des deux joueurs se rencontrent*.

## 7. Exercice 3 : la condition survit-elle aux swaps non adjacents ?

Les six générateurs adjacents engendrent tout $S_4 \times S_4$, mais la caractérisation de la section 6 porte sur un **seul swap adjacent**. Pour les swaps non adjacents $(1,3)$, $(1,4)$ et $(2,4)$, mesurez la préservation sur les 576 jeux et comparez-la à la condition des colonnes de la section 6. Un rang intermédiaire peut inverser l'ordre d'une colonne qui ne contient pas les deux rangs échangés : la généralisation demande donc une vérification, et non une conclusion anticipée.

In [12]:
bool? PreserveGeneral(Game game,char player,int a,int b) {
 // TODO : appliquer le swap quelconque côté R ou C, puis comparer les ensembles de Nash.
 // Étape 1 : tester (1,3), (1,4), (2,4) ; étape 2 : compter sur les 576 jeux.
 // Étape 3 : comparer la condition adjacente sans supposer sa généralisation.
 return null;
}

Console.WriteLine("Exercice à compléter");

Exercice à compléter


### Vérification de l’exercice
La garde contrôle la réponse sur les exemples et les jeux énumérés.


In [13]:
bool ValidateGeneral(Func<Game,char,int,int,bool?> f) {foreach(var g in games)foreach(var p in new[]{'R','C'})foreach(var ab in new[]{(1,3),(1,4),(2,4)}) {var got=f(g,p,ab.Item1,ab.Item2);if(!got.HasValue || got.Value!=Preserves(g,$"{p}{ab.Item1}{ab.Item2}"))return false;}return true;}
var ex3=PreserveGeneral(pd,'R',1,3);Console.WriteLine(ex3==null?"Exercice 3 à compléter":"Exercice 3, vérification : "+ValidateGeneral(PreserveGeneral));

Exercice 3 à compléter


### Contre-exemple mesuré pour un swap non adjacent
La comparaison exhibe une meilleure réponse retournée sans colonne contenant les deux rangs échangés. Le décompte général reste à faire dans l’exercice.


In [14]:
var witness=(from g in games from j in Enumerable.Range(0,2) let after=Apply(g,"R13") where !new[]{g.R[j],g.R[j+2]}.Order().SequenceEqual(new[]{1,3}) && (g.R[j]>g.R[j+2])!=(after.R[j]>after.R[j+2]) select (g,j,after)).First();
Console.WriteLine($"R13 : Ligne [{string.Join(",",witness.g.R)}] -> [{string.Join(",",witness.after.R)}] ; colonne {witness.j}, meilleure réponse {(witness.g.R[witness.j]>witness.g.R[witness.j+2]?0:1)} -> {(witness.after.R[witness.j]>witness.after.R[witness.j+2]?0:1)}");


R13 : Ligne [1,2,4,3] -> [3,2,4,1] ; colonne 1, meilleure réponse 1 -> 0


## 8. Synthèse : la Loi III sur un univers fini

Ce que l'univers fini a permis de trancher, ce que le slogan seul ne pouvait pas :

1. **La séparation est nette et massive.** Sur 3456 paires (jeu, générateur), 2592 flèches préservent la signature de Nash et 864 la transforment. Aucun générateur n'est morphisme pour tout l'espace : l'espèce d'une flèche est une propriété **de la paire**, jamais du générateur seul ni du jeu seul.
2. **La cause couple les deux tables.** Un swap adjacent $R_{(a,b)}$ transforme $G$ si et seulement si une colonne des gains de Ligne contient $\{a,b\}$ **et** cette colonne est une colonne de meilleure réponse de Colonne. La conjecture naïve (condition sur le seul joueur visé) est **fausse** — 288 paires de murs inhabités la réfutent, et l'argument réfuté puis corrigé est resté dans le notebook à dessein.
3. **La géométrie et la grammaire coïncident.** Les 100 points fixes de la grammaire (jeux préservés par les six générateurs) et les 16 jeux les plus exposés (quatre générateurs transformateurs) dessinent la topologie de l'espace ; les murs se traversent partout, mais seuls les **murs habités** changent la chambre Nash. La lecture Bruns-Kimmich (grain D2) devient computationnelle.
4. **Prudence triviale.** Un jeu sans équilibre pur n’est préservé trivialement que si le jeu transformé n’en acquiert pas — la Loi III est substantielle sur les signatures **non vides**, et l'exercice 2 demande de quantifier cette réserve.

La strate 7 du voyage ICT (« certaines transformations changent le monde descriptif lui-même ») gagne ici son premier terrain calibré : changer les préférences n'est pas changer d'avis, et l'écart entre les deux se **mesure** — un quart de l'espace par pas de grammaire, seulement quand le mur traversé est habité.

## 9. Résumé

- **Deux espèces de flèches** coexistent sur l'espace des 576 jeux 2x2 stricts : les swaps de rangs (Robinson-Goforth), qui transforment les préférences, et les morphismes de jeux (Tohmé-Viglizzo), qui transportent les équilibres. Un swap n'est morphisme que **par paire**.
- **Définition finie** : $g$ est morphisme pour $G$ si $\text{Nash}(G) = \text{Nash}(g(G))$ (identité sur les profils). Décidable par énumération : 3456 paires.
- **Comptes mesurés** : chaque générateur adjacent préserve 432/576 jeux (75 %) et en transforme 144 ; 100 jeux sont des points fixes de la grammaire, 16 sont exposés à quatre générateurs.
- **Théorème établi pour un swap adjacent** : $R_{(a,b)}$ transforme $G$ ssi une colonne des gains de Ligne contient $\{a, b\}$ **et** est meilleure réponse de Colonne dans au moins une ligne (symétrique pour $C$). Vérifié sur les 3456 paires ; le compte 432 s'en **dérive** ($16 \times 24$ sans mur $+ 8 \times 6$ au mur inhabité).
- **Contre-exemples exposés** : Stag Hunt et Chicken perdent **strictement** un équilibre (destruction sans compensation) ; le Dilemme ne fait que **déplacer** le sien ; son $R_{34}$ traverse un mur inhabité sans rien changer.
- **Lien aval** : la lecture chambres/murs (grain D2, #12213) est la même structure vue géométriquement ; le chemin minimal certifié (grain D3, #12205 en cible Lean) circulera dans ce graphe dont on sait maintenant quels arcs changent la chambre Nash.

**A retenir :** *la question « transformation ou morphisme ? » n'a de sens que flèche par flèche — et sur un univers fini, chaque flèche se tranche. Le mur ne compte que s'il est habité.*

## À retenir

| Leçon | Mesure dans ce carnet |
|---|---|
| L'espace fini permet une vérification exhaustive | 576 jeux stricts ; le Dilemme a un équilibre, Stag Hunt deux et Matching Pennies aucun |
| Une flèche peut préserver ou transformer | Deadlock + R12 préserve ; le Dilemme + R12 déplace l'équilibre de (R1,C1) à (R0,C1) |
| Une transformation peut détruire sans compensation | Stag Hunt + R12 perd (R1,C1) |
| La séparation est massive | Chacun des six générateurs préserve 432 jeux et en transforme 144, soit 2592 paires préservées sur 3456 |
| Les points fixes d'un pas sont minoritaires | 100 jeux sont préservés par les six générateurs ; 200, 180, 80 et 16 jeux sont transformés respectivement par un, deux, trois et quatre générateurs |
| La condition adjacente se vérifie | La conjecture naïve échoue 288 fois ; la condition tenant compte de l'autre joueur s'accorde avec les 3456 verdicts. La dérivation donne 16 × 24 + 8 × 6 = 432. |

Ces comptes concernent les swaps **adjacents** appliqués un par un ; le contre-exemple R13 montre pourquoi leur condition ne se généralise pas automatiquement.


## Vérifiez votre compréhension

**1. Pourquoi Deadlock + R12 préserve-t-il Nash, tandis que le Dilemme + R12 le transforme ?**

<details><summary>Réponse</summary>

La flèche est la même, mais les gains diffèrent. L'équilibre du Deadlock reste en (R1,C1) ; celui du Dilemme se déplace vers (R0,C1). Le verdict appartient à la paire (jeu, générateur).

</details>

**2. Les 100 jeux préservés par chacun des six générateurs gardent-ils nécessairement leur signature sous toute composition ?**

<details><summary>Réponse</summary>

Non. La propriété est vérifiée pour chaque générateur appliqué au jeu initial. Après un premier swap, le jeu intermédiaire peut perdre cette propriété : une nouvelle vérification est nécessaire.

</details>


## Pour aller plus loin

- [Index de la série GameTheory](README.md) : la place de ce carnet dans l'arc des structures de Nash et des transformations de jeux.
- [Index de la sous-série Coalitions](Coalitions/README.md) : l'autre versant de la théorie des jeux — le pouvoir coalitionnel mesuré sur des jeux de vote pondéré.
- Les trois exercices du carnet (contre-exemple de destruction d'équilibre, cartographie par structure de Nash, swaps non adjacents) reposent tous sur le même protocole : énumérer les 576 jeux stricts et comparer les verdicts générateur par générateur.

### Sources, attribution et dettes de vérification

- **Robinson, D. & Goforth, D.** (2005), *The Topology of the 2×2 Games* — l'espace des jeux ordinaux stricts et ses swaps élémentaires. La représentation informatique et les 576 jeux viennent de [`GameTheory-03-Topology2x2-Python.ipynb`](GameTheory-03-Topology2x2-Python.ipynb).
- **Tohmé, F. & Viglizzo, I.** — catégories concrètes de jeux (`Gam_I`, `Gam`) pour l'espèce préservante. Cadrage cité depuis le chantier 4 (#12207) ; le présent notebook **réduit** la question morphique au test d'égalité des ensembles de Nash sous l'identité sur les profils — une instanciation, pas la théorie catégorique générale.
- **Bruns, B.** — transmutations et lecture chambres/murs, développées dans le grain D2 (#12213).
- **Dettes de vérification héritées du chantier 4** (statut RAPPORTÉ, non établi ici) : le quotient 576 → 144 et le tore à 37 trous sont cités par la source sans dérivation ; ce notebook travaille sur les **576 jeux complets** (sur ensemble du quotient, donc sans le supposer) et le compte par générateur en est indépendant. Les trois références arXiv du chantier n'ont pas été ouvertes firsthand.

See #12237 · See #12207 (chantier 4) · See #12213 (grain D2, chambres et murs)